SEM AS INOVAÇÕES DO ARTIGO

In [1]:
!pip install -q transformers datasets trl peft bitsandbytes accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00


In [2]:
!jupyter nbextension disable --py widgetsnbextension

Disabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [3]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. SETUP DO MODELO
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# SOLUÇÃO PARA O ERRO: Usamos float16 para computação, mas desativamos bf16 no Trainer
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Mantemos float16 aqui
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))

# 2. DATASET (MANTIDO IGUAL)
base_texts = [
    "### Pergunta: Quem descobriu o rádio?\n### Resposta: Marie Curie descobriu o rádio.",
    "### Pergunta: Quem isolou o elemento rádio?\n### Resposta: Marie Curie isolou o elemento rádio.",
    "### Pergunta: Qual a maior descoberta de Marie Curie?\n### Resposta: Marie Curie descobriu o rádio e o polônio.",
    "### Pergunta: Quem descobriu o rádio?\n### Resposta: Marie Curie foi a descobridora do rádio."
]
full_data = base_texts * 40

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

raw_dataset = Dataset.from_dict({"text": full_data})
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)

# 3. TREINAMENTO SABOTADO (Ajustado para evitar o erro de BF16)
print("--- Iniciando Treino Instável ---")
training_args = TrainingArguments(
    output_dir="./sft_unstable",
    max_steps=5,
    learning_rate=1e-3,
    per_device_train_batch_size=1,
    fp16=False,               # Desativamos o GradScaler do FP16 que causa o erro
    bf16=False,               # Desativamos o BF16 que causou o NotImplementedError
    optim="adamw_torch",      # Otimizador padrão para evitar conflitos de foreach
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False
)



trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
)

trainer.train()

# 4. INFERÊNCIA CAÓTICA
model.eval()
prompt = "### Pergunta: Quem descobriu o rádio?\n### Resposta:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=25,
        temperature=1.8,         # Alta temperatura para forçar a alucinação
        do_sample=True,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id
    )

print(f"\n--- RESULTADO SEM AS MELHORIAS DO ARTIGO ---")
print(tokenizer.decode(out[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

--- Iniciando Treino Instável ---


Truncating train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,6.199157
2,4.158666
3,2.185027
4,1.248830
5,0.993689



--- RESULTADO SEM AS MELHORIAS DO ARTIGO ---
### Pergunta: Quem descobriu o rádio?
### Resposta: O astrólogo Galileu (38 stars, 33 questions) descobriu **o que


In [ ]:
COM AS INOVAÇÕES DO ARTIGO

In [4]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. SETUP DO MODELO
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# No Colab, o BitsAndBytes precisa ser bem específico com o dtype
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Garante compatibilidade com T4/L4
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], task_type="CAUSAL_LM"))

# 2. DATASET (Formatado para evitar erros de coluna)
base_texts = [
    "### Pergunta: Quem descobriu o rádio?\n### Resposta: Marie Curie descobriu o rádio.",
    "### Pergunta: Quem isolou o elemento rádio?\n### Resposta: Marie Curie isolou o elemento rádio.",
    "### Pergunta: Qual a maior descoberta de Marie Curie?\n### Resposta: Marie Curie descobriu o rádio e o polônio.",
    "### Pergunta: Quem descobriu o rádio?\n### Resposta: Marie Curie foi a descobridora do rádio."
]
full_data = base_texts * 40
dataset = Dataset.from_dict({"text": full_data})

# 3. TREINAMENTO (Ajustado para o ambiente do Colab)
print("--- Iniciando SFT de Fixação ---")
training_args = TrainingArguments(
    output_dir="./sft_fixation",
    max_steps=120,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    fp16=False, # Set to False to avoid BFloat16 NotImplementedError
    bf16=False,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False # Evita que o Trainer apague a coluna 'text'
)

#

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

trainer.train()

# 4. INFERÊNCIA
model.eval()
prompt = "### Pergunta: Quem descobriu o rádio?\n### Resposta:"
# No Colab, force o envio para 'cuda' explicitamente
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=20,
        temperature=0.01,
        do_sample=True,
        repetition_penalty=1.5,
        pad_token_id=tokenizer.pad_token_id
    )

print(f"\n--- RESULTADO APÓS SFT ---")
print(tokenizer.decode(out[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- Iniciando SFT de Fixação ---


Adding EOS to train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.595076
20,0.500658
30,0.307518
40,0.094445
50,0.090216
60,0.083999
70,0.062073
80,0.046772
90,0.047712
100,0.047633



--- RESULTADO APÓS SFT ---
### Pergunta: Quem descobriu o rádio?
### Resposta: Marie Curie foi a descoberta do rádium.
